# T-E2 AI Agent自动生成股市周报 — 数据获取

| 项目   | 内容 |
|--------|------|
| 课程   | 数据分析与经济决策（ds2026） |
| 题目   | T-E2：AI_Agent自动生成股市周报 |
| 小组   | 第 09 组 |
| 成员   | 梁志鹏（25210177）、吴璇子（25210264）、黄悦（25210145）、王鹤洋（25210243）、柯骋（25210150）、罗天（25210207）|
| GitHub | https://github.com/liangzhipeng82138-tech/ds2026-G09-ai_agent_stock_weekly_report |
| Pages  | https://liangzhipeng82138-tech.github.io/ds2026-G09-ai_agent_stock_weekly_report/ |
| 日期   | 2026-05-15 |

## 任务说明

本 Notebook 负责从在线数据源自动获取 A 股核心数据，包括：
1. 六大核心指数的日线行情（上证指数、深证成指、创业板指、科创50、沪深300、中证500）
2. 全市场涨跌家数、涨跌停数量
3. 北向资金流向数据（沪股通/深股通）
4. 两市成交额数据

数据获取后保存至 `data_raw/` 目录，不做任何修改。

In [ ]:
# 安装依赖（如未安装）
# !pip install akshare yfinance pandas

In [ ]:
import akshare as ak
import pandas as pd
import json
import os
from datetime import datetime, timedelta

print(f'akshare version: {ak.__version__}')
print(f'当前时间: {datetime.now()}')

# 创建原始数据目录
os.makedirs('../data_raw', exist_ok=True)

### 1. 获取六大核心指数日线行情

使用 `akshare` 的 `stock_zh_index_daily` 接口获取主要指数近两周的日线数据，用于计算周涨跌幅和振幅。

In [ ]:
# A股核心指数代码映射
INDEX_MAP = {
    '上证指数': 'sh000001',
    '深证成指': 'sz399001',
    '创业板指': 'sz399006',
    '科创50':  'sh000688',
    '沪深300': 'sh000300',
    '中证500': 'sh000905',
}

index_data = {}

for name, code in INDEX_MAP.items():
    try:
        print(f'正在获取 {name} ({code}) ...')
        df = ak.stock_zh_index_daily(symbol=code)
        # 取最近15个交易日（覆盖两周）
        df = df.tail(15).copy()
        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)
        index_data[name] = df
        print(f'  ✓ {name}: 获取到 {len(df)} 条记录, 最新日期 {df["date"].iloc[-1].strftime("%Y-%m-%d")}')
    except Exception as e:
        print(f'  ✗ {name} 获取失败: {e}')

print(f'\n成功获取 {len(index_data)} 个指数数据')

In [ ]:
# 预览上证指数数据
if '上证指数' in index_data:
    print(index_data['上证指数'].tail(10))

### 2. 保存指数原始数据

将每个指数的原始数据保存为 CSV 文件至 `data_raw/` 目录。

In [ ]:
# 保存原始指数数据
for name, df in index_data.items():
    filename = f'../data_raw/index_{name}.csv'
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f'已保存: {filename}')

### 3. 获取全市场涨跌家数数据

使用 akshare 获取全市场涨跌家数统计，用于分析市场情绪。

In [ ]:
# 获取全市场涨跌家数
try:
    print('正在获取全市场涨跌家数 ...')
    # 使用 stock_zh_a_spot_em 接口获取全市场实时行情
    spot_df = ak.stock_zh_a_spot_em()
    print(f'  ✓ 获取到 {len(spot_df)} 只股票数据')
    print(f'  列名: {list(spot_df.columns)}')
    
    # 保存原始数据
    spot_df.to_csv('../data_raw/stock_spot_all.csv', index=False, encoding='utf-8-sig')
    print('  已保存: ../data_raw/stock_spot_all.csv')
except Exception as e:
    print(f'  ✗ 获取失败: {e}')

### 4. 获取北向资金数据

北向资金是A股重要的外资指标，包括沪股通和深股通净流入数据。

In [ ]:
# 获取北向资金数据
try:
    print('正在获取北向资金数据 ...')
    # 沪股通+深股通合并 = 北向资金
    north_df = ak.stock_hsgt_north_net_flow_in_em(symbol='北向')
    north_df = north_df.tail(15).copy()
    print(f'  ✓ 获取到 {len(north_df)} 条北向资金记录')
    print(f'  列名: {list(north_df.columns)}')
    
    north_df.to_csv('../data_raw/northbound_flow.csv', index=False, encoding='utf-8-sig')
    print('  已保存: ../data_raw/northbound_flow.csv')
except Exception as e:
    print(f'  ✗ 北向资金获取失败: {e}')
    print('  尝试使用替代接口 ...')
    try:
        # 替代接口
        north_df = ak.stock_em_hsgt_north_net_flow_in(indicator='北向')
        north_df = north_df.tail(15).copy()
        north_df.to_csv('../data_raw/northbound_flow.csv', index=False, encoding='utf-8-sig')
        print(f'  ✓ 替代接口成功, 获取到 {len(north_df)} 条记录')
    except Exception as e2:
        print(f'  ✗ 替代接口也失败: {e2}')

### 5. 获取两市成交额数据

从上证和深证指数的日线数据中提取成交额信息。

In [ ]:
# 从指数数据中提取成交额
# 上证指数和深证成指的日线数据中包含成交额字段
turnover_data = {}

for idx_name in ['上证指数', '深证成指']:
    if idx_name in index_data:
        df = index_data[idx_name]
        if 'volume' in df.columns:
            turnover_data[idx_name] = df[['date', 'volume']].copy()
            print(f'{idx_name} 成交额数据: {len(df)} 条')
        else:
            print(f'{idx_name} 无 volume 字段, 可用字段: {list(df.columns)}')

# 合并两市成交额
if turnover_data:
    turnover_combined = pd.DataFrame()
    for name, df in turnover_data.items():
        temp = df.copy()
        temp.columns = ['date', f'{name}_volume']
        if turnover_combined.empty:
            turnover_combined = temp
        else:
            turnover_combined = turnover_combined.merge(temp, on='date', how='outer')
    
    turnover_combined['两市合计'] = turnover_combined.iloc[:, 1:].sum(axis=1)
    turnover_combined.to_csv('../data_raw/turnover.csv', index=False, encoding='utf-8-sig')
    print(f'\n已保存成交额数据: {len(turnover_combined)} 条')

### 6. 获取美股主要指数数据

使用 yfinance 获取美股三大指数（道琼斯、标普500、纳斯达克）作为外围市场参考。

In [ ]:
import yfinance as yf

US_INDEX = {
    '道琼斯': '^DJI',
    '标普500': '^GSPC',
    '纳斯达克': '^IXIC',
}

us_data = {}
for name, ticker in US_INDEX.items():
    try:
        print(f'正在获取 {name} ({ticker}) ...')
        df = yf.download(ticker, period='15d', progress=False)
        if not df.empty:
            us_data[name] = df
            print(f'  ✓ {name}: {len(df)} 条记录')
        else:
            print(f'  ✗ {name}: 无数据')
    except Exception as e:
        print(f'  ✗ {name} 获取失败: {e}')

# 保存美股数据
for name, df in us_data.items():
    filename = f'../data_raw/us_{name}.csv'
    df.to_csv(filename, encoding='utf-8-sig')
    print(f'已保存: {filename}')

### 7. 数据获取汇总

检查 `data_raw/` 目录下所有已获取的数据文件。